# Phase 2: Dual-Path Sequential Naive Bayes & Meta-Multinomial Stacking
**Project**: AURA - Adaptive User Risk Analytics
**Dataset**: Open University Learning Analytics Dataset (OULAD)
**Target Execution**: Google Colab (CPU)

---

### Objectives
1. **Pipeline Verification**: Load raw OULAD data or ingest via Phase 1 pipeline.
2. **Cartesian Grid Reshaping**: Format dynamic clicks and milestone scores into (N, 40) tensors aligned with static demographics.
3. **Dual-Path Naive Bayes Training**: Fit class-conditional Gaussian distributions on the training split with variance smoothing.
4. **Milestone Weight Optimization**: Calibrate milestone temporal weights W using validation log-loss grid search.
5. **Trajectory Evaluation**: Compute sequential posterior distributions week-by-week with temporal freezing upon withdrawal.
6. **Meta-Classifier Stacking**: Train multinomial logistic regression fusing dynamic Bayesian posteriors with static demographics at cutoff weeks 4, 8, 12, and 39.
7. **Validation & Benchmark Comparison**: Measure Macro F1, Per-Class Recall, and ROC-AUC against Phase 1 baselines.

In [ ]:
# Install dependencies in Colab environment
!pip install -q kagglehub polars pyarrow scikit-learn seaborn matplotlib scipy

import os
import sys
import shutil
import time
from pathlib import Path

if "/content" not in sys.path:
    sys.path.insert(0, "/content")
if "." not in sys.path:
    sys.path.append(".")

import polars as pl
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import f1_score, roc_auc_score, recall_score, classification_report

from src.data_loader import run_pipeline, TARGET_MAP, RAW_SCHEMAS
from src.dual_path_nb import DualPathNaiveBayes, default_milestone_weight_array, calibrate_milestone_weights
from src.meta_classifier import MetaClassifierWrapper, stack_features
from src.evaluation import evaluate_multiclass_predictions

pl.enable_string_cache()
np.random.seed(42)

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
FIG_DIR = Path("figures")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Environment and modules initialized successfully.")

In [ ]:
# Ingest raw OULAD tables via kagglehub if not present
import kagglehub

REQUIRED_TABLES = [
    "assessments.csv",
    "courses.csv",
    "studentAssessment.csv",
    "studentInfo.csv",
    "studentRegistration.csv",
    "studentVle.csv",
    "vle.csv",
]

missing_tables = [t for t in REQUIRED_TABLES if not (RAW_DIR / t).exists() or (RAW_DIR / t).stat().st_size == 0]

if missing_tables:
    print(f"Downloading OULAD dataset via kagglehub (missing: {missing_tables})...")
    slugs = [
        "anlgrbz/student-demographics-online-education-dataoulad",
        "rocki37/open-university-learning-analytics-dataset",
    ]
    download_dir = None
    for slug in slugs:
        try:
            print(f"Trying slug: {slug}...")
            download_dir = kagglehub.dataset_download(slug)
            print(f"Downloaded to: {download_dir}")
            break
        except Exception as e:
            print(f"Slug {slug} download failed: {e}")

    if download_dir and Path(download_dir).exists():
        for root, _, files in os.walk(download_dir):
            for f in files:
                for req in REQUIRED_TABLES:
                    if f.lower() == req.lower():
                        dest = RAW_DIR / req
                        shutil.copy2(os.path.join(root, f), dest)
                        print(f"Copied {f} -> {dest}")

# Verify all tables are present
for t in REQUIRED_TABLES:
    assert (RAW_DIR / t).exists(), f"Missing required raw table: {t}"
print("All raw OULAD tables present and verified.")

In [ ]:
# Run Phase 1 pipeline if processed artifacts do not already exist
req_processed = [
    "dynamic_features_weekly.parquet",
    "static_demographics_raw.parquet",
    "encoded_demographics.npy",
    "cohort_splits.npz",
]
missing_processed = [f for f in req_processed if not (PROCESSED_DIR / f).exists()]

if missing_processed:
    print(f"Executing Phase 1 ingestion pipeline for missing artifacts: {missing_processed}...")
    t0 = time.perf_counter()
    run_pipeline(RAW_DIR, PROCESSED_DIR)
    print(f"Phase 1 pipeline generated in {time.perf_counter() - t0:.2f}s")
else:
    print("Processed artifacts already exist. Skipping pipeline execution.")

In [ ]:
# Load processed artifacts
print("Loading processed artifacts into memory...")
dynamic_df = pl.read_parquet(PROCESSED_DIR / "dynamic_features_weekly.parquet")
base_students_df = pl.read_parquet(PROCESSED_DIR / "static_demographics_raw.parquet")
encoded_demographics = np.load(PROCESSED_DIR / "encoded_demographics.npy")
splits = np.load(PROCESSED_DIR / "cohort_splits.npz", allow_pickle=True)

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

n_students = base_students_df.shape[0]
print(f"Total unique students: {n_students:,}")
print(f"Train cohort: {len(train_idx):,} | Val cohort: {len(val_idx):,} | Test cohort: {len(test_idx):,}")
print(f"Encoded demographic matrix shape: {encoded_demographics.shape}")

In [ ]:
# Reshape Cartesian grid into aligned tensors (N, 40)
print("Pivoting dynamic Cartesian grid into (N, 40) tensors...")
t0 = time.perf_counter()

# Construct student index map from base_students_df
student_keys = base_students_df.select([
    "code_module", "code_presentation", "id_student"
]).with_row_index("student_idx")

joined_dynamic = dynamic_df.join(
    student_keys, on=["code_module", "code_presentation", "id_student"], how="inner"
).sort(["student_idx", "week"])

# Pivot clicks and scores
clicks_pivot = joined_dynamic.pivot(on="week", index="student_idx", values="log_clicks").sort("student_idx")
scores_pivot = joined_dynamic.pivot(on="week", index="student_idx", values="normalized_score").sort("student_idx")

week_cols = [c for c in clicks_pivot.columns if c != "student_idx"]
# Sort week columns numerically (0..39)
week_cols = sorted(week_cols, key=lambda x: int(x))

X_clicks = clicks_pivot.select(week_cols).to_numpy().astype(np.float64)
X_scores = scores_pivot.select(week_cols).to_numpy().astype(np.float64)

# Extract unregistration weeks, student IDs, and labels
student_ids = base_students_df["id_student"].to_numpy()
unreg_dates = base_students_df["date_unregistration"].to_numpy()
unreg_weeks = np.where(
    np.isnan(unreg_dates.astype(float)),
    -1.0,
    np.where(unreg_dates < 0, 0.0, np.floor(unreg_dates / 7.0) + 1.0)
)

targets = base_students_df["final_result"].to_pandas().map(TARGET_MAP).to_numpy(dtype=np.int64)

print(f"X_clicks shape: {X_clicks.shape}")
print(f"X_scores shape: {X_scores.shape} (NaN count: {np.isnan(X_scores).sum():,})")
print(f"Tensors materialized in {time.perf_counter() - t0:.2f}s")

In [ ]:
# Fit Dual-Path Naive Bayes on Train Split
print("Fitting Dual-Path Naive Bayes model on training cohort...")
t0 = time.perf_counter()

X_clicks_train = X_clicks[train_idx]
X_scores_train = X_scores[train_idx]
y_train = targets[train_idx]

nb_model = DualPathNaiveBayes(num_classes=4, var_smoothing=1e-4)
nb_model.fit(X_clicks_train, X_scores_train, y_train)

print(f"Model fitted in {time.perf_counter() - t0:.3f}s")
print("Class empirical priors:")
inv_map = {v: k for k, v in TARGET_MAP.items()}
for k in range(4):
    print(f"  Class {k} ({inv_map[k]}): {nb_model.priors_[k]:.4f}")

In [ ]:
# Calibrate Milestone Weights on Validation Cohort
print("Calibrating milestone temporal weights W via validation grid search...")
t0 = time.perf_counter()

X_clicks_val = X_clicks[val_idx]
X_scores_val = X_scores[val_idx]
y_val = targets[val_idx]

grid_params = {
    "assessment_weeks": [3, 8, 19, 24],
    "base_weights": [0.5, 0.8, 1.0],
    "milestone_weights": [1.0, 1.5, 2.0, 2.5, 3.0],
    "eval_cutoff_week": 39,
}

best_weights, best_config = calibrate_milestone_weights(
    val_clicks=X_clicks_val,
    val_scores=X_scores_val,
    val_labels=y_val,
    grid_search_params=grid_params,
    num_classes=4,
)

print(f"Grid search completed in {time.perf_counter() - t0:.2f}s")
print(f"Best configuration: {best_config}")

# Compare log-loss of Uniform (W=1) vs Calibrated Milestone Weights
uniform_weights = np.ones(40, dtype=np.float64)
preds_uniform = nb_model.predict_terminal_posteriors(X_clicks_val, X_scores_val, cutoff_week=39, weights=uniform_weights)
preds_calibrated = nb_model.predict_terminal_posteriors(X_clicks_val, X_scores_val, cutoff_week=39, weights=best_weights)

loss_uniform = -np.mean(np.log(np.clip(preds_uniform[np.arange(len(y_val)), y_val], 1e-15, 1.0)))
loss_calibrated = -np.mean(np.log(np.clip(preds_calibrated[np.arange(len(y_val)), y_val], 1e-15, 1.0)))

print(f"Validation Log-Loss: Uniform W=1: {loss_uniform:.4f} | Calibrated W: {loss_calibrated:.4f}")

In [ ]:
# Compute Sequential Posterior Trajectories Across All Weeks
print("Generating full posterior probability trajectories (N, 40, 4) with temporal withdrawal freezing...")
t0 = time.perf_counter()

all_posteriors = nb_model.predict_weekly_posteriors(
    weekly_clicks=X_clicks,
    weekly_scores=X_scores,
    weights=best_weights,
    unregistration_weeks=unreg_weeks,
)

print(f"Posteriors computed in {time.perf_counter() - t0:.2f}s, shape: {all_posteriors.shape}")

In [ ]:
# Stacking Meta-Classifier: Train & Evaluate at Cutoff Weeks [4, 8, 12, 39]
cutoffs = [4, 8, 12, 39]
class_names = ["Pass", "Withdrawn", "Fail", "Distinction"]
results_summary = []

print("=" * 70)
print("PHASE 2 / 4 MULTI-CUTOFF STACKING META-CLASSIFIER BENCHMARK")
print("=" * 70)

for cutoff in cutoffs:
    # Extract Bayesian posteriors at cutoff week
    P_train = all_posteriors[train_idx, cutoff, :]
    P_val = all_posteriors[val_idx, cutoff, :]

    # Stack with static demographic vector
    X_train_stacked = stack_features(P_train, encoded_demographics[train_idx])
    X_val_stacked = stack_features(P_val, encoded_demographics[val_idx])

    # Fit balanced Multinomial Logistic Regression meta-learner
    meta = MetaClassifierWrapper(C=1.0, max_iter=1000, random_state=42)
    meta.fit(X_train_stacked, y_train)

    # Predict on validation set
    y_pred = meta.predict(X_val_stacked)
    y_proba = meta.predict_proba(X_val_stacked)

    # Compute metrics
    macro_f1 = f1_score(y_val, y_pred, average="macro")
    weighted_f1 = f1_score(y_val, y_pred, average="weighted")
    ovr_auc = roc_auc_score(y_val, y_proba, multi_class="ovr")
    recall_per_class = recall_score(y_val, y_pred, average=None)

    # Also evaluate pure Bayesian cutoff (without demographics)
    bayesian_pred = np.argmax(P_val, axis=1)
    bayesian_macro_f1 = f1_score(y_val, bayesian_pred, average="macro")
    bayesian_auc = roc_auc_score(y_val, P_val, multi_class="ovr")

    print(f"\n>>> Cutoff Week {cutoff:02d} <<<")
    print(f"  Pure Bayesian : Macro F1: {bayesian_macro_f1:.4f} | ROC-AUC: {bayesian_auc:.4f}")
    print(f"  Stacked Model : Macro F1: {macro_f1:.4f} | Weighted F1: {weighted_f1:.4f} | ROC-AUC: {ovr_auc:.4f}")
    print("  Per-Class Recall (Pass, Withdrawn, Fail, Distinction):")
    for c_idx, c_name in enumerate(class_names):
        print(f"    {c_name:<12}: {recall_per_class[c_idx]:.4f}")

    results_summary.append({
        "cutoff_week": cutoff,
        "bayesian_macro_f1": bayesian_macro_f1,
        "bayesian_ovr_auc": bayesian_auc,
        "stacked_macro_f1": macro_f1,
        "stacked_weighted_f1": weighted_f1,
        "stacked_ovr_auc": ovr_auc,
        "recall_pass": recall_per_class[0],
        "recall_withdrawn": recall_per_class[1],
        "recall_fail": recall_per_class[2],
        "recall_distinction": recall_per_class[3],
    })

results_df = pd.DataFrame(results_summary)
print("\nSummary Table:")
print(results_df.to_string(index=False))

In [ ]:
# Visualizations: Plot Trajectories and Cutoff Performance
# 1. Macro F1 & AUC trajectory progression across cutoffs
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

ax1.plot(results_df["cutoff_week"], results_df["stacked_macro_f1"], "b-o", label="Stacked Macro F1", linewidth=2)
ax1.plot(results_df["cutoff_week"], results_df["bayesian_macro_f1"], "b--s", label="Bayesian Macro F1", linewidth=2)
ax2.plot(results_df["cutoff_week"], results_df["stacked_ovr_auc"], "r-o", label="Stacked ROC-AUC", linewidth=2)
ax2.plot(results_df["cutoff_week"], results_df["bayesian_ovr_auc"], "r--s", label="Bayesian ROC-AUC", linewidth=2)

ax1.set_xlabel("Course Cutoff Week", fontsize=12)
ax1.set_ylabel("Macro F1 Score", color="b", fontsize=12)
ax2.set_ylabel("One-vs-Rest ROC-AUC", color="r", fontsize=12)
plt.title("AURA Multi-Cutoff Early Warning Performance Progression", fontsize=14)
ax1.grid(True, linestyle="--", alpha=0.5)
fig.tight_layout()
plt.savefig(FIG_DIR / "early_warning_progression.png", dpi=150)
plt.close()
print("Saved early_warning_progression.png")

# 2. Plot Sample Student Risk Trajectories
sample_indices = [val_idx[0], val_idx[1], val_idx[2]]
fig, axes = plt.subplots(len(sample_indices), 1, figsize=(10, 3 * len(sample_indices)), sharex=True)

weeks = np.arange(40)
colors = ["#2ecc71", "#e74c3c", "#f39c12", "#3498db"]

for ax, s_idx in zip(axes, sample_indices):
    student_probs = all_posteriors[s_idx, :, :]  # (40, 4)
    true_label = class_names[targets[s_idx]]
    unreg = unreg_weeks[s_idx]
    
    for k_class in range(4):
        ax.plot(weeks, student_probs[:, k_class], label=class_names[k_class], color=colors[k_class], linewidth=2)
    
    if unreg >= 0:
        ax.axvline(x=unreg, color="black", linestyle=":", label=f"Unregistered (Wk {int(unreg)})")
    
    ax.set_ylabel("Posterior P(Y=k)", fontsize=10)
    student_id = student_ids[s_idx]
    ax.set_title(f"Student ID: {student_id} | True Outcome: {true_label}", fontsize=11)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.4)

axes[-1].set_xlabel("Course Timeline Week (t)", fontsize=12)
fig.tight_layout()
plt.savefig(FIG_DIR / "sample_student_trajectories.png", dpi=150)
plt.close()
print("Saved sample_student_trajectories.png")

In [ ]:
# Persist Trained Artifacts and Metrics
print("Persisting Phase 2 model metrics and calibrated weights...")
np.savez_compressed(
    PROCESSED_DIR / "phase2_model_metrics.npz",
    best_weights=best_weights,
    best_config=np.array([list(best_config.items())], dtype=object),
    results_summary=results_df.to_dict(orient="records"),
)
print("Phase 2 Model Training & Benchmark Pipeline Completed Successfully.")